# 05 · Scaling the replay with PySpark

**Question:** the order book of one stock never depends on another, so can the replay run per stock in parallel, and is it still right?

Every ITCH order message carries the stock locate, and a replace keeps the order in the same stock. So the events can be grouped by locate and each group replayed independently with the same `replay` function (`itch_forensics.scaled`). Two things to establish: the parallel version gives **identical results** to the single-process one, and **what it costs or saves** at this size.

In [1]:
import json, subprocess, sys, time, pathlib
import numpy as np
import pandas as pd
from itch_forensics.book import ANOMALIES, build_events, replay
from itch_forensics.decode import decode_all
from itch_forensics.framing import index_messages

pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)
buf = pathlib.Path("../data/itch.bin").read_bytes()
events = build_events(decode_all(buf, index_messages(buf)))
cpu = next((l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "unknown")
import os; print(f"{len(events):,} events | {os.cpu_count()} logical cores | {cpu}")

7,397,062 events | 2 logical cores | Intel(R) Xeon(R) Processor @ 2.80GHz


## Timings

Nothing else is running during these. The single process tracks the best bid/offer for **every** stock, as the Spark version does, so they do the same work. Spark runs as a separate local-mode process with 1 and then 2 cores. One run each, so read them as an order of magnitude.

In [2]:
t = time.time(); bad, _, bbo_all, state = replay(events, events.locate.unique().tolist()); single_s = time.time() - t
timings = [{"method": "single process", "cores": 1, "seconds": round(single_s, 1), "events_per_s": round(len(events) / single_s)}]
for cores in ("1", "2"):
    out = subprocess.run([sys.executable, "-m", "itch_forensics.scaled", "--parquet", "../data/parquet", "--cores", cores],
                         capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
    r = json.loads(out)
    timings.append({"method": "PySpark by stock", "cores": int(cores), "seconds": r["seconds"], "events_per_s": r["events_per_s"]})
timings = pd.DataFrame(timings); timings

,method,cores,seconds,events_per_s
0,single process,1,43.1,171530
1,PySpark by stock,1,100.1,73894
2,PySpark by stock,2,62.9,117593


In [3]:
one, two = timings.seconds[1], timings.seconds[2]
print(f"Spark, 1 core -> 2 cores: {one / two:.2f}x faster")
print(f"Spark on 2 cores vs the single process: {timings.seconds[0] / two:.2f}x ({'faster' if two < timings.seconds[0] else 'slower'})")

Spark, 1 core -> 2 cores: 1.59x faster
Spark on 2 cores vs the single process: 0.69x (slower)


At this size Spark is **not** a win on two cores. Starting the JVM, reading Parquet, shuffling by stock and moving each group to a Python worker costs more than the parallelism saves. It does scale (the 1 to 2 core ratio above), so with enough cores or enough data it would overtake the single process, but this notebook cannot show where that crossing is.

The reason to want the per-stock design is not speed here but that it makes the work divisible: the single-process version holds every live order of every stock in one dictionary, and the per-stock version never holds more than one stock's book. (Memory was not measured.)

## Is the parallel version right?

Per stock, for all stocks: live orders, resting shares, executed shares, best-bid/offer change count, and every anomaly counter.

In [4]:
from itch_forensics.spark import get_spark, register
from itch_forensics.scaled import replay_by_stock
spark = get_spark("nb05", shuffle_partitions=16); register(spark, "../data/parquet")
par = replay_by_stock(spark).set_index("locate")
bl = state["by_locate"].set_index("locate")
idx_all = par.index
live_s = bl.live_orders.reindex(idx_all).fillna(0).astype(int); rest_s = bl.resting_shares.reindex(idx_all).fillna(0).astype(int)
exec_s = pd.Series(state["executed_by_locate"]).reindex(idx_all).fillna(0).astype(int)
bbo_s = bbo_all.groupby("locate").size().reindex(idx_all).fillna(0).astype(int)
checks = pd.DataFrame({
    "stocks compared": [len(par)] * 6,
    "identical": [int((par.live_orders == live_s).sum()), int((par.resting_shares_orders == rest_s).sum()),
                  int((par.executed_shares == exec_s).sum()), int((par.bbo_changes == bbo_s).sum()),
                  int((par.resting_shares_orders == par.resting_shares_levels).sum()),
                  int((par[list(ANOMALIES)].sum(axis=1) == 0).sum())]},
    index=["live orders", "resting shares", "executed shares", "best bid/offer changes", "orders = price levels", "no anomalies"])
checks

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=45241 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=45241 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/10 20:47:11 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/10/10 20:47:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/10/10 20:47:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/sql/pandas/typehints.py:505: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/sql/pandas/functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,stocks compared,identical
live orders,8890,8890
resting shares,8890,8890
executed shares,8890,8890
best bid/offer changes,8890,8890
orders = price levels,8890,8890
no anomalies,8890,8890


In [5]:
print(f"totals: {int(par.events.sum()):,} events, {int(par.live_orders.sum()):,} live orders, "
      f"{int(par.resting_shares_orders.sum()):,} resting shares, {int(par.bbo_changes.sum()):,} best bid/offer changes")
print("single process:", f"{len(events):,}", f"{state['live_orders']:,}", f"{state['resting_shares_orders']:,}", f"{len(bbo_all):,}")
assert (checks.identical == checks["stocks compared"]).all()
spark.stop()

totals: 7,397,062 events, 1,538,626 live orders, 879,432,660 resting shares, 1,292,708 best bid/offer changes
single process: 7,397,062 1,538,626 879,432,660 1,292,708


Every stock agrees on every measure, so the parallel replay is a faithful, divisible version of the single-process one.

**Limits.** This relies on an order's later messages carrying the same locate as its add. The single-process replay counts violations (`locate_mismatch`) and finds none here; a violation would show up in the parallel version as an unknown reference. It was shown equivalent on 7.4 million events from a 300 MB prefix, not on a full day.